# Pol-ISARSpaceTarget: OBB Augmentation and Backbone Study

Purpose:

- Rebuild a duplicate-free 1314/162/162 dataset before training.
- Compare YOLO11m-OBB with augmentation disabled and enabled.
- Optionally compare YOLO26m-OBB when that OBB checkpoint is supported.
- Save train/validation losses, learning rate, P/R/F1 and mAP for the PPT.

This notebook changes one factor at a time. The primary comparison is the
augmentation ablation on the same YOLO11m-OBB backbone.

Main input variant:

- `single_vv`, because previous experiments showed it had the strongest mAP.

This notebook deliberately changes only model capacity. Polarimetric fusion
and augmentation ablations should remain separate experiments.



In [ ]:
# Colab may already have Pillow loaded when pip updates it, leaving mixed PIL
# modules in memory. Install a consistent wheel and restart exactly once.
import os
import subprocess
import sys
from pathlib import Path

DEPS_MARKER = Path("/content/.polisar_obb_aug_dependencies_ready")
if not DEPS_MARKER.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            "--force-reinstall",
            "--no-deps",
            "Pillow==12.3.0",
        ]
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"])
    DEPS_MARKER.write_text("ready\n")
    print("Dependencies installed. Restarting the Colab kernel once; then run all cells again.")
    os.kill(os.getpid(), 9)
else:
    print("Dependency setup already completed for this runtime.")



In [ ]:
import json
import os
import re
import shutil
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from ultralytics import YOLO

print("Ultralytics OBB docs: https://docs.ultralytics.com/tasks/obb/")



## Runtime Check



In [ ]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("GPU is not connected. In Colab select Runtime > Change runtime type > T4 GPU.")




## Paths and Dataset Root



In [ ]:
try:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        DRIVE_BASE = Path("/content/drive/MyDrive")
    except Exception as exc:
        print("Mount at /content/drive failed:", exc)
        print("Trying alternate mount point: /content/gdrive")
        drive.mount("/content/gdrive")
        DRIVE_BASE = Path("/content/gdrive/MyDrive")
except Exception as exc:
    print("Drive mount skipped:", exc)
    DRIVE_BASE = Path("/content/drive/MyDrive")

ROOT_CANDIDATES = [
    DRIVE_BASE / "drdo" / "Pol_ISARSpaceTarget1.0",
    DRIVE_BASE / "drdo" / "Pol_ISARSpaceTarget1.0" / "Pol_ISARSpaceTarget1.0",
    DRIVE_BASE / "drdo" / "Pol_ISAR",
    DRIVE_BASE / "drdo" / "Pol_ISAR" / "Pol_ISARSpaceTarget1.0",
    Path("/content/Pol_ISARSpaceTarget1.0"),
    Path("/content/Pol_ISAR"),
    Path("C:/Users/keerthish/Desktop/drdo/Pol_ISARSpaceTarget1.0"),
]


def is_valid_polisar_root(path: Path) -> bool:
    return (path / "ImageSets" / "labelTxt").exists() and (path / "ImageSets" / "images").exists()


DATASET_ROOT = next((p for p in ROOT_CANDIDATES if is_valid_polisar_root(p)), None)
if DATASET_ROOT is None:
    print("Could not find dataset in default locations. Searching under Drive drdo folder...")
    search_roots = [DRIVE_BASE / "drdo", Path("/content")]
    found = []
    for root in search_roots:
        if root.exists():
            found.extend([p for p in root.rglob("ImageSets") if (p / "labelTxt").exists() and (p / "images").exists()])
    if found:
        DATASET_ROOT = found[0].parent
    else:
        raise FileNotFoundError(
            "Could not locate Pol-ISAR dataset root. Expected a folder containing ImageSets/labelTxt and ImageSets/images."
        )
print("DATASET_ROOT:", DATASET_ROOT)

LABELTXT_DIR = DATASET_ROOT / "ImageSets" / "labelTxt"
DAT_DIR = DATASET_ROOT / "ImageSets" / "images"
TIFF_DIR = DATASET_ROOT / "TIFFImages" / "AmplitudeImages"

assert LABELTXT_DIR.exists(), f"Missing labels: {LABELTXT_DIR}"
assert DAT_DIR.exists(), f"Missing complex dat files: {DAT_DIR}"

OUT_ROOT = Path("/content/polisar_obb_yolo")
DRIVE_OUT = DRIVE_BASE / "drdo" / "polisar_obb_augmentation_backbone"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)




## Label Reader and Class Summary



In [ ]:
def read_dota_label_file(path: Path) -> List[Tuple[List[float], str, int]]:
    rows = []
    if not path.exists() or not path.read_text().strip():
        return rows
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 10:
            continue
        coords = [float(x) for x in parts[:8]]
        cls_name = parts[8]
        difficult = int(float(parts[9]))
        rows.append((coords, cls_name, difficult))
    return rows


class_names = sorted(
    {
        cls
        for split in ["train", "valid", "test"]
        for txt in (LABELTXT_DIR / split).glob("*.txt")
        for _, cls, _ in read_dota_label_file(txt)
    }
)
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("Class map:", class_to_id)

split_summary = []
for split in ["train", "valid", "test"]:
    txt_files = sorted((LABELTXT_DIR / split).glob("*.txt"))
    obj_count = sum(len(read_dota_label_file(p)) for p in txt_files)
    split_summary.append({"split": split, "images": len(txt_files), "objects": obj_count})
split_summary_df = pd.DataFrame(split_summary)
display(split_summary_df)
expected_counts = {"train": 1314, "valid": 162, "test": 162}
actual_counts = dict(zip(split_summary_df["split"], split_summary_df["images"]))
if actual_counts != expected_counts:
    print("Raw Drive label counts contain duplicates:", actual_counts)
    print("Canonical deduplication is applied during conversion below.")
split_summary_df.to_csv(DRIVE_OUT / "dataset_split_summary.csv", index=False)
(DRIVE_OUT / "obb_class_names.json").write_text(json.dumps(class_to_id, indent=2))




## Image Builders

Only the strongest previous input, `single_vv`, is built. This keeps the
experiment focused on detector size and avoids rebuilding unused variants.



In [ ]:
POL_SUFFIXES = ["HH", "HV", "VH", "VV"]


def robust_uint8(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = np.log1p(np.maximum(x, 0.0))
    lo, hi = np.percentile(x, [1, 99])
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (255 * x).astype(np.uint8)


def base_name_candidates(base_name: str) -> List[str]:
    cleaned = re.sub(r"\s+\(\d+\)$", "", base_name)
    return [base_name, cleaned] if cleaned != base_name else [base_name]


def find_first_existing(paths: Sequence[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def read_complex_dat(path: Path, shape: Tuple[int, int] = (256, 256)) -> np.ndarray:
    raw = np.fromfile(path, dtype=np.float32)
    real = raw[0::2].reshape(shape)
    imag = raw[1::2].reshape(shape)
    return real + 1j * imag


def find_dat(base_name: str, pol: str, split: str) -> Path:
    path = find_first_existing([DAT_DIR / split / f"{candidate}_{pol}.dat" for candidate in base_name_candidates(base_name)])
    if path is None:
        raise FileNotFoundError(f"No {pol}.dat found for {base_name} in split {split}")
    return path


def build_single_vv(base_name: str, split: str) -> np.ndarray:
    tiff_path = find_first_existing([TIFF_DIR / f"{candidate}_VV.tiff" for candidate in base_name_candidates(base_name)])
    if tiff_path is not None:
        img = np.array(Image.open(tiff_path).convert("L"))
        return np.stack([img, img, img], axis=-1)
    vv = read_complex_dat(find_dat(base_name, "VV", split))
    img = robust_uint8(np.abs(vv))
    return np.stack([img, img, img], axis=-1)


def build_pauli_fusion(base_name: str, split: str) -> np.ndarray:
    c = {pol: read_complex_dat(find_dat(base_name, pol, split)) for pol in POL_SUFFIXES}
    hh, hv, vh, vv = c["HH"], c["HV"], c["VH"], c["VV"]
    r = robust_uint8(np.abs(hh - vv))
    g = robust_uint8(np.abs(hv + vh))
    b = robust_uint8(np.abs(hh + vv))
    return np.stack([r, g, b], axis=-1)




## Convert DOTA-Style OBB Labels to Ultralytics YOLO-OBB Format

Existing labels:

`x1 y1 x2 y2 x3 y3 x4 y4 class difficult`

Ultralytics OBB labels:

`class x1 y1 x2 y2 x3 y3 x4 y4`

with normalized corner coordinates in `[0, 1]`.



In [ ]:
def convert_label_to_yolo_obb(src_txt: Path, dst_txt: Path, width: int = 256, height: int = 256) -> int:
    out_lines = []
    for coords, cls_name, difficult in read_dota_label_file(src_txt):
        if difficult == 1:
            continue
        cls_id = class_to_id[cls_name]
        norm = []
        for idx, value in enumerate(coords):
            norm.append(value / width if idx % 2 == 0 else value / height)
        norm = [min(max(v, 0.0), 1.0) for v in norm]
        out_lines.append(" ".join([str(cls_id)] + [f"{v:.6f}" for v in norm]))
    dst_txt.parent.mkdir(parents=True, exist_ok=True)
    dst_txt.write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    return len(out_lines)


def write_data_yaml(out_dir: Path) -> Path:
    data_yaml = {
        "path": str(out_dir),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": {idx: name for idx, name in enumerate(class_names)},
    }
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
    return yaml_path


def yolo_dataset_is_complete(out_dir: Path) -> bool:
    yaml_path = out_dir / "data.yaml"
    return (
        yaml_path.exists()
        and (out_dir / "images" / "train").exists()
        and any((out_dir / "images" / "train").glob("*.png"))
        and (out_dir / "labels" / "train").exists()
        and any((out_dir / "labels" / "train").glob("*.txt"))
    )


def canonical_label_files(split: str) -> List[Path]:
    """Keep one label per physical image and prefer the unsuffixed filename."""
    selected: Dict[str, Path] = {}
    for path in sorted((LABELTXT_DIR / split).glob("*.txt")):
        canonical = re.sub(r"\s+\(\d+\)$", "", path.stem)
        current = selected.get(canonical)
        if current is None or path.stem == canonical:
            selected[canonical] = path
    files = [selected[key] for key in sorted(selected)]
    expected = {"train": 1314, "valid": 162, "test": 162}[split]
    if len(files) != expected:
        raise AssertionError(f"{split}: expected {expected} unique labels, found {len(files)}")
    return files


def build_yolo_dataset(variant: str, max_per_split: int | None = None, force_rebuild: bool = False) -> Path:
    out_dir = OUT_ROOT / variant
    if yolo_dataset_is_complete(out_dir) and not force_rebuild:
        print("Using existing dataset:", out_dir)
        return out_dir / "data.yaml"

    if out_dir.exists():
        shutil.rmtree(out_dir)
    for split in ["train", "valid", "test"]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        txt_files = canonical_label_files(split)
        if max_per_split is not None:
            txt_files = txt_files[:max_per_split]

        for i, txt in enumerate(txt_files, start=1):
            base = re.sub(r"\s+\(\d+\)$", "", txt.stem)
            if variant == "single_vv":
                img = build_single_vv(base, split)
            elif variant == "pauli_fusion":
                img = build_pauli_fusion(base, split)
            else:
                raise ValueError(f"Cannot rebuild unknown variant: {variant}")

            cv2.imwrite(str(out_dir / "images" / split / f"{base}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            convert_label_to_yolo_obb(txt, out_dir / "labels" / split / f"{base}.txt")
            if i % 100 == 0:
                print(variant, split, i, "/", len(txt_files))

    yaml_path = write_data_yaml(out_dir)
    print("Prepared:", out_dir)
    return yaml_path




## Build or Reuse Datasets



In [ ]:
MAX_PER_SPLIT = None
# Mandatory once because previous cached data contained 174 rather than 162 test labels.
FORCE_REBUILD_DATASETS = True

dataset_yamls = {
    "single_vv": build_yolo_dataset("single_vv", max_per_split=MAX_PER_SPLIT, force_rebuild=FORCE_REBUILD_DATASETS),
}

print(dataset_yamls)




## Controlled Experiment List

Official Ultralytics OBB models use the `-obb.pt` suffix and are pretrained on
DOTA-style oriented boxes. If one model is unavailable in your installed
version, this notebook logs the failure and continues with the rest.



In [ ]:
EXPERIMENTS = [
    {"method": "YOLO11m-OBB", "weights": "yolo11m-obb.pt", "batch": 8, "augmentation": "none"},
    {"method": "YOLO11m-OBB", "weights": "yolo11m-obb.pt", "batch": 8, "augmentation": "obb_safe"},
]

# Set True only after the two controlled YOLO11m runs finish. The code first
# checks that the requested OBB checkpoint can be loaded by this Ultralytics release.
RUN_OPTIONAL_YOLO26 = False
if RUN_OPTIONAL_YOLO26:
    EXPERIMENTS.append(
        {"method": "YOLO26m-OBB", "weights": "yolo26m-obb.pt", "batch": 8, "augmentation": "obb_safe"}
    )

# Main comparison: compare OBB methods on strongest previous input.
INPUT_VARIANTS_TO_RUN = ["single_vv"]

# Optional: uncomment to run the full grid. This takes longer.
# INPUT_VARIANTS_TO_RUN = ["single_vv", "pauli_fusion", "learned_fusion"]

EPOCHS = 100
IMGSZ = 256
PATIENCE = 100
SAVE_PERIOD = 1
SEED = 42

print("Experiments:", EXPERIMENTS)
print("Input variants:", INPUT_VARIANTS_TO_RUN)
print(f"Training setting: max epochs={EPOCHS}, patience={PATIENCE}, save_period={SAVE_PERIOD}, seed={SEED}")




## Train and Evaluate One Method



In [ ]:
def metric_value(metrics, *names):
    for name in names:
        obj = metrics
        ok = True
        for part in name.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            try:
                return float(obj)
            except Exception:
                pass
    return None


def train_eval_obb(method_name: str, weights: str, batch: int, augmentation: str, variant: str, yaml_path: Path) -> Dict:
    run_name = f"{method_name}_{variant}_{augmentation}".replace("/", "_").replace(" ", "_")
    run_dir = DRIVE_OUT / "runs_obb_methods" / run_name
    last_ckpt = run_dir / "weights" / "last.pt"
    best_ckpt = run_dir / "weights" / "best.pt"
    completed_flag = run_dir / "training_complete.txt"
    started = time.time()
    row = {
        "method": method_name,
        "weights": weights,
        "input_variant": variant,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": batch,
        "augmentation": augmentation,
        "status": "started",
        "run_name": run_name,
    }
    try:
        if completed_flag.exists() and best_ckpt.exists():
            print(f"Training already complete for {run_name}; evaluating saved best checkpoint.")
            model = YOLO(str(best_ckpt))
        elif last_ckpt.exists():
            print(f"Resuming {run_name} from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            model.train(resume=True)
            completed_flag.write_text("Training completed successfully.\n")
        else:
            model = YOLO(weights)
            use_aug = augmentation == "obb_safe"
            model.train(
                data=str(yaml_path),
                task="obb",
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=batch,
                project=str(DRIVE_OUT / "runs_obb_methods"),
                name=run_name,
                exist_ok=True,
                patience=PATIENCE,
                save=True,
                save_period=SAVE_PERIOD,
                plots=True,
                cache=False,
                workers=2,
                amp=True,
                seed=SEED,
                deterministic=True,
                degrees=10.0 if use_aug else 0.0,
                translate=0.08 if use_aug else 0.0,
                scale=0.25 if use_aug else 0.0,
                fliplr=0.5 if use_aug else 0.0,
                flipud=0.5 if use_aug else 0.0,
                mosaic=0.5 if use_aug else 0.0,
                mixup=0.0,
                close_mosaic=10,
                device=DEVICE,
            )
            completed_flag.write_text("Training completed successfully.\n")
        eval_ckpt = best_ckpt if best_ckpt.exists() else last_ckpt
        eval_model = YOLO(str(eval_ckpt)) if eval_ckpt.exists() else model
        metrics = eval_model.val(data=str(yaml_path), split="test", task="obb", imgsz=IMGSZ, plots=True)
        elapsed = time.time() - started
        row.update(
            {
                "status": "ok",
                "test_precision": metric_value(metrics, "box.mp"),
                "test_recall": metric_value(metrics, "box.mr"),
                "test_mAP50": metric_value(metrics, "box.map50"),
                "test_mAP50_95": metric_value(metrics, "box.map"),
                "elapsed_min": elapsed / 60,
                "run_dir": str(run_dir),
                "best_checkpoint": str(best_ckpt),
                "last_checkpoint": str(last_ckpt),
            }
        )
        p, r = row["test_precision"], row["test_recall"]
        row["test_f1"] = 2 * p * r / (p + r) if p is not None and r is not None and p + r else None
    except Exception as exc:
        row.update({"status": "failed", "error": repr(exc), "elapsed_min": (time.time() - started) / 60})
        print("FAILED:", method_name, variant, exc)
    return row




## Run Comparison

This may take several hours for 100 epochs.

Runtime-disconnect handling:

- Checkpoints are saved every epoch in Google Drive.
- If Colab disconnects, reconnect GPU and rerun the notebook.
- The training function checks for `weights/last.pt` and resumes automatically.
- Keep `INPUT_VARIANTS_TO_RUN = ["single_vv"]` first, because it gave the best
  previous performance and keeps runtime manageable.

For a quick smoke test, set `EPOCHS = 2` and `MAX_PER_SPLIT = 50`, then restart
runtime and rerun from the top.



In [ ]:
results = []
for variant in INPUT_VARIANTS_TO_RUN:
    if variant not in dataset_yamls:
        print("Skipping missing variant:", variant)
        continue
    for method in EXPERIMENTS:
        row = train_eval_obb(
            method["method"], method["weights"], method["batch"], method["augmentation"], variant, dataset_yamls[variant]
        )
        results.append(row)
        pd.DataFrame(results).to_csv(DRIVE_OUT / "obb_method_comparison_live.csv", index=False)
        print(json.dumps(row, indent=2))

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(DRIVE_OUT / "obb_method_comparison_final.csv", index=False)
(DRIVE_OUT / "obb_method_comparison_final.json").write_text(json.dumps(results, indent=2))




## Plot Comparison



In [ ]:
ok_df = results_df[results_df["status"] == "ok"].copy()
if len(ok_df):
    ok_df = ok_df.sort_values("test_mAP50_95", ascending=False)
    display(ok_df[["method", "input_variant", "test_precision", "test_recall", "test_mAP50", "test_mAP50_95", "elapsed_min"]])

    labels = ok_df["method"] + "\n" + ok_df["augmentation"]
    x = np.arange(len(ok_df)); width = 0.18
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for offset, col, title in [(-1.5, "test_precision", "Precision"), (-0.5, "test_recall", "Recall"),
                               (0.5, "test_f1", "F1"), (1.5, "test_mAP50_95", "mAP50-95")]:
        ax.bar(x + offset * width, ok_df[col], width, label=title)
    ax.set_xticks(x, labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Held-Out OBB Comparison")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(ncol=4)

    fig.tight_layout()
    fig.savefig(DRIVE_OUT / "obb_precision_recall_f1_map_comparison.png", dpi=180)
    plt.show()
else:
    print("No completed runs to plot yet.")




## Baseline Reference From Earlier Experiments



In [ ]:
reference = pd.DataFrame(
    [
        {
            "method": "YOLO11n-OBB",
            "input_variant": "single_vv",
            "test_precision": 0.804,
            "test_recall": 0.819,
            "test_mAP50": 0.821,
            "test_mAP50_95": 0.582,
            "source": "previous completed notebook",
        },
        {
            "method": "YOLO11n-OBB",
            "input_variant": "pauli_fusion",
            "test_precision": 0.828,
            "test_recall": 0.739,
            "test_mAP50": 0.783,
            "test_mAP50_95": 0.573,
            "source": "previous completed notebook",
        },
        {
            "method": "YOLO11n-OBB",
            "input_variant": "learned_fusion",
            "test_precision": 0.770,
            "test_recall": 0.683,
            "test_mAP50": 0.743,
            "test_mAP50_95": 0.518,
            "source": "previous completed notebook",
        },
    ]
)
reference["test_f1"] = 2 * reference["test_precision"] * reference["test_recall"] / (
    reference["test_precision"] + reference["test_recall"]
)
display(reference)
reference.to_csv(DRIVE_OUT / "previous_obb_reference_results.csv", index=False)




## Final Model-Scale Table and Training Curves

The earlier `n` result is a historical reference. The `s` and `m` rows are
directly comparable to each other because this notebook uses one fixed setup.



In [ ]:
new_cols = [
    "method",
    "input_variant",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_mAP50",
    "test_mAP50_95",
]
historical = reference[new_cols].copy()
historical["comparison_type"] = "earlier baseline"

current = ok_df[new_cols].copy() if len(ok_df) else pd.DataFrame(columns=new_cols)
current["comparison_type"] = "controlled larger-model run"
model_scale_comparison = pd.concat([historical.iloc[[0]], current], ignore_index=True)
display(model_scale_comparison)
model_scale_comparison.to_csv(DRIVE_OUT / "model_scale_comparison.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plotted = False
for method in EXPERIMENTS:
    run_name = f"{method['method']}_single_vv_{method['augmentation']}".replace("/", "_").replace(" ", "_")
    csv_path = DRIVE_OUT / "runs_obb_methods" / run_name / "results.csv"
    if not csv_path.exists():
        continue
    history = pd.read_csv(csv_path)
    history.columns = [c.strip() for c in history.columns]
    epoch = history["epoch"] + 1
    train_loss_col = next((c for c in history.columns if c.startswith("train/") and "box_loss" in c), None)
    val_loss_col = next((c for c in history.columns if c.startswith("val/") and "box_loss" in c), None)
    map_col = next((c for c in history.columns if "metrics/mAP50-95" in c), None)
    lr_col = next((c for c in history.columns if c.startswith("lr/")), None)
    label = f"{method['method']} / {method['augmentation']}"
    if train_loss_col:
        axes[0, 0].plot(epoch, history[train_loss_col], label=label)
        plotted = True
    if val_loss_col:
        axes[0, 1].plot(epoch, history[val_loss_col], label=label)
        plotted = True
    if map_col:
        axes[1, 0].plot(epoch, history[map_col], label=label)
    if lr_col:
        axes[1, 1].plot(epoch, history[lr_col], label=label)

if plotted:
    axes[0, 0].set(title="Training Box Loss", xlabel="Epoch", ylabel="Loss")
    axes[0, 1].set(title="Validation Box Loss", xlabel="Epoch", ylabel="Loss")
    axes[1, 0].set(title="Validation mAP50-95", xlabel="Epoch", ylabel="mAP50-95", ylim=(0, 1))
    axes[1, 1].set(title="Learning Rate", xlabel="Epoch", ylabel="LR")
    for ax in axes.ravel():
        ax.grid(alpha=0.25)
        ax.legend()
    fig.tight_layout()
    fig.savefig(DRIVE_OUT / "augmentation_backbone_epoch_curves.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    plt.close(fig)
    print("Training curves will appear after at least one model finishes training.")
